In [1]:
# from scipy.io import loadmat
#
# misc_root = "archive/misc"
# mat_data = loadmat(os.path.join(misc_root, "make_model_name.mat"))
# print(mat_data.keys())
#
# car_type_mat = loadmat(os.path.join(misc_root, "car_type.mat"))
# print(car_type_mat.keys())


In [2]:
# import os
# from collections import defaultdict
#
# # Analyser la structure des dossiers d'images
# print("=== ANALYSE DE LA STRUCTURE DES DONNÉES ===")
#
# # Compter le nombre de modèles par marque
# make_model_count = defaultdict(set)
#
# for make_id in range(1, 20):  # Analyser les 20 premières marques
#     image_dir = os.path.join("archive/image", str(make_id))
#     if not os.path.exists(image_dir):
#         continue
#
#     print(f"\nMake ID {make_id}:")
#     for subdir in os.listdir(image_dir):
#         subdir_path = os.path.join(image_dir, subdir)
#         if os.path.isdir(subdir_path):
#             print(f"  {subdir}/")
#             # Essayer d'extraire le modèle du nom du sous-dossier
#             make_model_count[make_id].add(subdir)
#
# print(f"\nModèles trouvés par marque:")
# for make_id, models in make_model_count.items():
#     print(f"Make {make_id}: {len(models)} modèles - {sorted(models)}")

In [3]:
# import os
# import re
# import shutil
# from scipy.io import loadmat
# import numpy as np
# from tqdm import tqdm
# import pandas as pd

# # --- chemins ---
# compcars_root = "archive"
# images_root = os.path.join(compcars_root, "image")
# labels_root = os.path.join(compcars_root, "label")
# misc_root = os.path.join(compcars_root, "misc")
# output_root = "dataset_corrected"
# os.makedirs(output_root, exist_ok=True)

# # --- utilitaires ---
# def sanitize(name: str) -> str:
#     name = name.strip().replace("/", "_").replace("\\", "_").replace(" ", "_")
#     name = re.sub(r"[^A-Za-z0-9_.\-]+", "_", name)
#     return name or "Unknown"

# def extract_matlab_string(cell):
#     if isinstance(cell, np.ndarray):
#         if cell.size == 0:
#             return None
#         item = cell.flat[0]
#         if isinstance(item, str):
#             return item
#         elif isinstance(item, np.ndarray):
#             return extract_matlab_string(item)
#     elif isinstance(cell, str):
#         return cell
#     return None

# # --- charge les noms de marques et modèles ---
# mat_path = os.path.join(misc_root, "make_model_name.mat")
# mat = loadmat(mat_path)
# model_names_mat = mat['model_names']
# make_names_mat = mat['make_names']

# make_names_list = []
# for i in range(make_names_mat.shape[0]):
#     make_name = extract_matlab_string(make_names_mat[i, 0])
#     if make_name and make_name.startswith("['") and make_name.endswith("']"):
#         make_name = make_name[2:-2]
#     make_names_list.append(make_name or f"Make_{i+1}")

# model_names_list = []
# for i in range(model_names_mat.shape[0]):
#     model_name = extract_matlab_string(model_names_mat[i, 0])
#     if model_name is None or model_name == '[]' or model_name == '':
#         model_names_list.append(f"Unknown_Model_{i+1}")
#     else:
#         if model_name.startswith("['") and model_name.endswith("']"):
#             model_name = model_name[2:-2]
#         model_names_list.append(model_name)

# # --- Lire le fichier attributes.txt pour le mapping ---
# print("Lecture du fichier attributes.txt...")
# attributes_path = os.path.join(misc_root, "attributes.txt")

# # Créer un mapping model_id -> make_id basé sur les attributs
# model_to_make = {}
# try:
#     with open(attributes_path, 'r', encoding='utf-8') as f:
#         lines = f.readlines()
#         # Saute la première ligne (en-tête)
#         for line in lines[1:]:
#             parts = line.strip().split()
#             if len(parts) >= 6:
#                 model_id = int(parts[0])
#                 # Le make_id peut être déduit de la structure ou d'autres fichiers
#                 # Pour l'instant, nous allons utiliser une approche différente
#                 model_to_make[model_id] = None  # À compléter plus tard
# except:
#     print("Impossible de lire attributes.txt")

# # --- APPROCHE ALTERNATIVE: Utiliser la structure des dossiers ---
# print("Création du mapping à partir de la structure...")

# # Analyser la structure pour trouver la correspondance
# make_model_mapping = {}

# for make_id in tqdm(range(1, len(make_names_list) + 1), desc="Analyse structure"):
#     make_dir = os.path.join(images_root, str(make_id))
#     if not os.path.exists(make_dir):
#         continue

#     # Compter les sous-dossiers (qui représentent les modèles)
#     subfolders = []
#     for item in os.listdir(make_dir):
#         item_path = os.path.join(make_dir, item)
#         if os.path.isdir(item_path):
#             subfolders.append(item)

#     if subfolders:
#         make_model_mapping[make_id] = {
#             'make_name': make_names_list[make_id - 1],
#             'subfolders': subfolders,
#             'model_count': len(subfolders)
#         }

# # Afficher la structure analysée
# print("\nStructure analysée:")
# for make_id, info in list(make_model_mapping.items())[:20]:
#     print(f"Make {make_id} ({info['make_name']}): {info['model_count']} modèles - {info['subfolders']}")

# # --- Indexation détaillée des images ---
# print("\nIndexation détaillée des images...")
# image_details = {}

# for make_id in tqdm(range(1, len(make_names_list) + 1), desc="Indexation détaillée"):
#     make_dir = os.path.join(images_root, str(make_id))
#     if not os.path.exists(make_dir):
#         continue

#     for root, dirs, files in os.walk(make_dir):
#         for file in files:
#             if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
#                 # Extraire le chemin relatif pour trouver le modèle
#                 rel_path = os.path.relpath(root, make_dir)
#                 subfolder = rel_path.split(os.sep)[0] if os.sep in rel_path else rel_path

#                 base_name = os.path.splitext(file)[0]
#                 key = (make_id, base_name)

#                 image_details[key] = {
#                     'path': os.path.join(root, file),
#                     'subfolder': subfolder,
#                     'make_id': make_id
#                 }

# print(f"Images indexées: {len(image_details)}")

# # --- Copie avec mapping basé sur la structure ---
# total_copied = 0
# per_class_counts = {}
# unknown_mappings = 0

# print("\nCopie avec mapping structurel...")
# for make_id in tqdm(range(1, len(make_names_list) + 1), desc="Traitement final"):
#     label_dir = os.path.join(labels_root, str(make_id))
#     if not os.path.exists(label_dir):
#         continue

#     make_name = make_names_list[make_id - 1]

#     for root, _, files in os.walk(label_dir):
#         for file in files:
#             if not file.endswith('.txt'):
#                 continue

#             txt_path = os.path.join(root, file)
#             base_name = os.path.splitext(file)[0]

#             image_key = (make_id, base_name)
#             if image_key not in image_details:
#                 continue

#             img_info = image_details[image_key]
#             img_path = img_info['path']
#             subfolder = img_info['subfolder']

#             try:
#                 with open(txt_path, 'r') as f:
#                     local_model_id = int(f.readline().strip())
#             except:
#                 continue

#             if local_model_id <= 0:
#                 continue

#             # UTILISER le sous-dossier comme nom de modèle
#             # C'est la clé : le sous-dossier EST le nom du modèle
#             model_name = subfolder

#             # Nettoyer le nom du modèle si nécessaire
#             if model_name.isdigit():
#                 # Si c'est un nombre, essayons de trouver un nom plus descriptif
#                 if make_id in make_model_mapping:
#                     subfolders = make_model_mapping[make_id]['subfolders']
#                     if int(model_name) <= len(subfolders):
#                         model_name = subfolders[int(model_name) - 1]
#                     else:
#                         model_name = f"Model_{model_name}"
#                 else:
#                     model_name = f"Model_{model_name}"

#             class_name = f"{make_name} {model_name}"
#             sanitized_name = sanitize(class_name)

#             dst_dir = os.path.join(output_root, sanitized_name)
#             os.makedirs(dst_dir, exist_ok=True)

#             dst_filename = f"{make_id}_{model_name}_{os.path.basename(img_path)}"
#             dst_path = os.path.join(dst_dir, dst_filename)

#             counter = 1
#             while os.path.exists(dst_path):
#                 name, ext = os.path.splitext(dst_filename)
#                 dst_path = os.path.join(dst_dir, f"{name}_{counter}{ext}")
#                 counter += 1

#             try:
#                 shutil.copy2(img_path, dst_path)
#                 total_copied += 1
#                 per_class_counts[sanitized_name] = per_class_counts.get(sanitized_name, 0) + 1

#                 if total_copied <= 20:
#                     print(f"  {make_name} {model_name}")

#             except Exception as e:
#                 print(f"Erreur: {e}")
#                 continue

# print(f"\n✅ COPIE TERMINÉE!")
# print(f"Images copiées: {total_copied}")
# print(f"Classes créées: {len(per_class_counts)}")
# print(f"Mappings inconnus: {unknown_mappings}")

# print("\nTop 20 classes:")
# sorted_classes = sorted(per_class_counts.items(), key=lambda x: x[1], reverse=True)
# for class_name, count in sorted_classes[:20]:
#     print(f"  {class_name}: {count} images")

# # Vérification
# print(f"\nVérification des dossiers:")
# output_dirs = [d for d in os.listdir(output_root) if os.path.isdir(os.path.join(output_root, d))]
# print(f"Dossiers créés: {len(output_dirs)}")
# print("Exemples:")
# for d in output_dirs[:10]:
#     print(f"  {d}")

In [4]:
# import os
# import shutil
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from tqdm import tqdm  # pip install tqdm

# # --- PARAMÈTRES ---
# DOSSIER_SOURCE = "compCars/CompCars/data/image"
# DOSSIER_DEST = "destination"
# MAX_WORKERS = 8
# # -----------------------------

# # Fonction pour extraire le nom du modèle à partir du chemin
# def extraire_modele(chemin):
#     # Exemple : compCars/image/Audi/A4L/2011/image1.jpg
#     segments = chemin.split(os.sep)
#     if len(segments) >= 4:
#         return segments[-3]  # Model = 3e segment depuis la fin
#     return None

# # Copier un fichier individuel
# def copier_fichier(src, dst):
#     os.makedirs(os.path.dirname(dst), exist_ok=True)
#     shutil.copy2(src, dst)
#     return src, dst

# # Copier tous les fichiers d'un dossier en parallèle
# def copier_fichiers_multithread(fichiers, dest_dir):
#     with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#         futures = {
#             executor.submit(copier_fichier, src, os.path.join(dest_dir, os.path.basename(src))): src
#             for src in fichiers
#         }
#         for future in as_completed(futures):
#             src, dst = future.result()
#             # print(f"Copié : {src} → {dst}")  # Décommenter si tu veux voir chaque fichier

# # Parcourir tout le dataset
# def copier_compcars_multithread(source_dir, dest_dir):
#     all_files = []

#     # Récupérer tous les fichiers avec leur chemin complet
#     for root, dirs, files in os.walk(source_dir):
#         for f in files:
#             all_files.append(os.path.join(root, f))

#     print(f"Nombre total de fichiers : {len(all_files)}")
    
#     # Traitement avec barre de progression
#     for f in tqdm(all_files, desc="Copie des fichiers"):
#         modele = extraire_modele(f)
#         if modele:
#             dest_modele = os.path.join(dest_dir, modele)
#             copier_fichiers_multithread([f], dest_modele)
#         else:
#             dest_modele = os.path.join(dest_dir, "Autres")
#             copier_fichiers_multithread([f], dest_modele)

# # --- Exécution ---
# if __name__ == "__main__":
#     copier_compcars_multithread(DOSSIER_SOURCE, DOSSIER_DEST)


Nombre total de fichiers : 136726


Copie des fichiers: 100%|██████████| 136726/136726 [2:08:29<00:00, 17.74it/s]  


In [5]:
# import os
# import pandas as pd

# # --- PARAMÈTRES ---
# root = "compCars/CompCars/data"        # dossier courant ou chemin complet
# max_depth = 5     # profondeur max à explorer
# max_rows = 200    # nombre maximum de dossiers à afficher
# # ------------------

# def depth_from_root(path, root):
#     rel = os.path.relpath(path, root)
#     if rel in ('.', ''):
#         return 0
#     return rel.count(os.sep) + 1

# data = []

# for dirpath, dirnames, filenames in os.walk(root):
#     depth = depth_from_root(dirpath, root)
#     if depth > max_depth:
#         dirnames[:] = []  # ne pas descendre plus profond
#         continue
#     data.append({
#         'relpath': os.path.relpath(dirpath, root),
#         'depth': depth,
#         'num_files': len([f for f in filenames if os.path.isfile(os.path.join(dirpath, f))]),
#         'num_subdirs': len(dirnames),
#         'sample_files': ';'.join(filenames[:5])
#     })
#     if len(data) >= max_rows:
#         break

# # Créer un DataFrame et l'afficher
# df = pd.DataFrame(data)
# df


In [2]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import scipy.io as sio
import numpy as np

# --- PARAMÈTRES ---
DOSSIER_SOURCE = "compCars/CompCars/data/image"
DOSSIER_DEST = "destination"
MAT_PATH = "compCars/CompCars/data/misc/make_model_name.mat"
MAX_WORKERS = 8
# -----------------------------

# Charger le mapping ID → nom lisible depuis le .mat
def charger_mapping(mat_path):
    data = sio.loadmat(mat_path)
    models = data.get("model_names", None)
    if models is None:
        raise ValueError("Clé 'model_names' introuvable dans le .mat")

    mapping = {}
    for i in range(models.shape[0]):
        model_entry = models[i,0]
        if isinstance(model_entry, np.ndarray) and model_entry.size > 0:
            model_name = str(model_entry[0])
            model_id = str(i+1)  # MATLAB commence à 1
            mapping[model_id] = model_name
        else:
            # cas où le modèle est vide
            mapping[str(i+1)] = f"Model_{i+1}"

    return mapping

# Copier tout un dossier
def copier_dossier(src_dir, dest_dir):
    os.makedirs(dest_dir, exist_ok=True)
    for f in os.listdir(src_dir):
        src = os.path.join(src_dir, f)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dest_dir, f))
    return src_dir, dest_dir

# Parcourir et copier
def parcourir_et_copier(source_dir, dest_dir, mapping):
    depth4_dirs = []
    for dirpath, dirnames, filenames in os.walk(source_dir):
        if len(filenames) > 0:
            rel = os.path.relpath(dirpath, source_dir)
            parts = rel.split(os.sep)
            # format attendu: Make / ModelID / Year
            if len(parts) == 3:
                model_id = parts[1]
                depth4_dirs.append((dirpath, model_id))

    print(f"Nombre de dossiers modèles trouvés : {len(depth4_dirs)}")

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for d, model_id in depth4_dirs:
            nom_modele = mapping.get(model_id, f"Model_{model_id}")
            dest_courant = os.path.join(dest_dir, nom_modele)
            futures.append(executor.submit(copier_dossier, d, dest_courant))

        for future in tqdm(as_completed(futures), total=len(futures), desc="Copie des modèles"):
            results.append(future.result())

    print("Copie terminée ✅")

# --- Exécution ---
if __name__ == "__main__":
    mapping = charger_mapping(MAT_PATH)
    parcourir_et_copier(DOSSIER_SOURCE, DOSSIER_DEST, mapping)


Nombre de dossiers modèles trouvés : 4446


Copie des modèles: 100%|██████████| 4446/4446 [07:10<00:00, 10.33it/s]

Copie terminée ✅


In [8]:
import scipy.io as sio

# Charger le fichier .mat
mat_path = "compCars/CompCars/data/misc/make_model_name.mat"
data = sio.loadmat(mat_path)

# Explorer les clés disponibles
print(data.keys())


dict_keys(['__header__', '__version__', '__globals__', 'make_names', 'model_names'])


In [12]:
import numpy as np

makes = data["make_names"]
models = data["model_names"]

# Exemple : afficher les 5 premiers modèles
for i in range(5):
    make = makes[i][0][0]
    model = models[i][0][0]
    print(f"{i+1} → {make} {model}")


1 → ABT Audi A3 hatchback
2 → BAC Audi A4L
3 → Conquest Audi A6L
4 → DS Audi Q3
5 → Dacia Audi Q5


In [18]:
import scipy.io as sio

mat_path = "compCars/CompCars/data/misc/make_model_name.mat"
data = sio.loadmat(mat_path)

print("Clés disponibles :", data.keys())

models = data.get("model_names", None)
if models is not None:
    print("Type de model_names :", type(models))
    print("Shape de model_names :", models.shape)
    print("Exemple brut :", models[:5])


Clés disponibles : dict_keys(['__header__', '__version__', '__globals__', 'make_names', 'model_names'])
Type de model_names : <class 'numpy.ndarray'>
Shape de model_names : (2004, 1)
Exemple brut : [[array(['Audi A3 hatchback'], dtype='<U17')]
 [array(['Audi A4L'], dtype='<U8')]
 [array(['Audi A6L'], dtype='<U8')]
 [array(['Audi Q3'], dtype='<U7')]
 [array(['Audi Q5'], dtype='<U7')]]
